# Behavior-Aware SASRec

Train and compare:
- **purchase_only**: SASRec on purchase sequences only
- **behavior_aware**: same SASRec backbone + cart add/remove behavior embeddings

**Comparability contract with `Step2_CandidateSampling_PopularityBaseline.ipynb`:**
1. Load Step1 `artifacts/sequences_and_splits.pkl` (global 14/14-day cutoffs, same users/targets/vocab)
2. Load Step2 `artifacts/shared_evaluation_candidates.pkl` (same 100-neg candidate sets)
3. Score those candidates and evaluate with `evaluation_utils.py` (HitRate@10, NDCG@10, MRR)
4. Target is always the shared next-purchase target; cart events are context only

Requires Step1 + Step2 artifacts before running.


## 0. Setup


In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import json
import pickle
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "artifacts" / "sequences_and_splits.pkl").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "artifacts" / "sequences_and_splits.pkl").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils import evaluate_model_dataset

print("Repo root:", REPO_ROOT.resolve())
print("Torch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")


Repo root: /content
Torch: 2.11.0+cu130 | device: cuda


## 1. Config

Paths and cutoffs come from the shared artifacts. Do not change `MAX_SEQ_LEN` / eval negatives independently of Step1–Step2.


In [2]:
# --- shared artifacts (must match popularity baseline) ---
ARTIFACT_DIR = REPO_ROOT / "artifacts"
SEQUENCES_PATH = ARTIFACT_DIR / "sequences_and_splits.pkl"
CANDIDATES_PATH = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"

# --- event data (for histories; cart used only by behavior_aware) ---
DATA_CANDIDATE_DIRS = [
    REPO_ROOT / "data" / "subsampled",
    Path("data/subsampled"),
    Path("/content/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]

# --- model / train ---
MODE = "both"  # "purchase_only" | "behavior_aware" | "both"
HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 20
N_NEGATIVES = 1          # training negatives only
PATIENCE = 5
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Cap training instances for memory/runtime; None = use all sliding-window buys
MAX_TRAIN_INSTANCES = None

OUTPUT_DIR = ARTIFACT_DIR / "sasrec_aligned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE =", DEVICE)
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())


DEVICE = cuda
OUTPUT_DIR = /content/artifacts/sasrec_aligned


## 2. Load shared Step1 / Step2 artifacts


In [3]:
with open(SEQUENCES_PATH, "rb") as f:
    split_data = pickle.load(f)

with open(CANDIDATES_PATH, "rb") as f:
    cand_data = pickle.load(f)

config = split_data["config"]
MAX_SEQ_LEN = int(config["MAX_SEQ_LEN"])
VAL_DAYS = int(config["VAL_DAYS"])
TEST_DAYS = int(config["TEST_DAYS"])
VAL_CUTOFF = pd.Timestamp(config["val_cutoff"])
TEST_CUTOFF = pd.Timestamp(config["test_cutoff"])

train_sequences = split_data["train_sequences"]
val_targets = split_data["val_targets"]
test_targets = split_data["test_targets"]
val_eval_users = list(split_data["val_eval_users"])
test_eval_users = list(split_data["test_eval_users"])
item_to_idx = {int(k): int(v) for k, v in split_data["item_to_idx"].items()}
idx_to_item = {int(k): int(v) for k, v in split_data["idx_to_item"].items()}
n_items = int(split_data["n_items"])

val_candidate_sets = cand_data["val_candidate_sets"]
test_candidate_sets = cand_data["test_candidate_sets"]
N_EVAL_NEGATIVES = int(cand_data["n_negatives"])
EVAL_K = int(cand_data.get("eval_k", 10))

assert MAX_SEQ_LEN == int(cand_data["max_seq_len"])
assert N_EVAL_NEGATIVES == 100
assert int(cand_data["random_seed"]) == SEED == 42
assert pd.Timestamp(cand_data["val_cutoff"]) == VAL_CUTOFF
assert pd.Timestamp(cand_data["test_cutoff"]) == TEST_CUTOFF
assert set(val_candidate_sets) == set(val_eval_users)
assert set(test_candidate_sets) == set(test_eval_users)

print("Shared artifacts loaded.")
print(f"MAX_SEQ_LEN={MAX_SEQ_LEN} | VAL_DAYS={VAL_DAYS} | TEST_DAYS={TEST_DAYS}")
print(f"Cutoffs: val={VAL_CUTOFF} | test={TEST_CUTOFF}")
print(f"Train users: {len(train_sequences):,} | items: {n_items:,}")
print(f"Val users: {len(val_eval_users):,} | test users: {len(test_eval_users):,}")
print(f"Candidates/user: {N_EVAL_NEGATIVES + 1} (shared with popularity)")


Shared artifacts loaded.
MAX_SEQ_LEN=50 | VAL_DAYS=14 | TEST_DAYS=14
Cutoffs: val=2022-11-10 00:09:15 | test=2022-11-24 00:09:15
Train users: 246,307 | items: 449,958
Val users: 34,127 | test users: 35,637
Candidates/user: 101 (shared with popularity)


## 3. Load interaction events

Purchases rebuild train/eval histories under the same cutoffs. Cart add/remove are used only as context for `behavior_aware`.


In [5]:
BEHAVIOR_PAD = 0
BEHAVIOR_BUY = 1
BEHAVIOR_ADD = 2
BEHAVIOR_REMOVE = 3

BEHAVIOR_NAME_TO_ID = {
    "product_buy": BEHAVIOR_BUY,
    "add_to_cart": BEHAVIOR_ADD,
    "remove_from_cart": BEHAVIOR_REMOVE,
}

DATA_DIR = next((p for p in DATA_CANDIDATE_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find subsampled parquet files. Expected one of: "
        + ", ".join(str(p) for p in DATA_CANDIDATE_DIRS)
    )
print("Loading events from:", DATA_DIR.resolve())

EVENT_FILES = {
    "product_buy": "product_buy_subsampled.parquet",
    "add_to_cart": "add_to_cart_subsampled.parquet",
    "remove_from_cart": "remove_from_cart_subsampled.parquet",
}

frames = []
for behavior, filename in EVENT_FILES.items():
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path).drop_duplicates().copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp", "client_id", "sku"])
    df["client_id"] = df["client_id"].astype(np.int64)
    df["sku"] = df["sku"].astype(np.int64)
    df["behavior"] = behavior
    frames.append(df[["client_id", "sku", "timestamp", "behavior"]])

events = pd.concat(frames, ignore_index=True)
events = events[events["sku"].isin(item_to_idx)].copy()
events["item_idx"] = events["sku"].map(item_to_idx).astype(np.int64)
events["behavior_id"] = events["behavior"].map(BEHAVIOR_NAME_TO_ID).astype(np.int64)
events = events.sort_values(["client_id", "timestamp", "behavior"]).reset_index(drop=True)

print(events["behavior"].value_counts())
print(f"Rows kept in shared vocab: {len(events):,}")


Loading events from: /content/subsampled
behavior
add_to_cart         3828537
remove_from_cart    1535464
product_buy         1171501
Name: count, dtype: int64
Rows kept in shared vocab: 6,535,502


## 4. Build training instances + eval histories (global temporal split)

Mirrors Step1:
- train purchases are strictly before `val_cutoff`
- val target = first purchase in `[val_cutoff, test_cutoff)`
- test target = first purchase in `[test_cutoff, end]`
- history for a target = events **before** that purchase timestamp (cart allowed for behavior_aware)


In [6]:
@dataclass(frozen=True)
class BehaviorEvent:
    item_id: int  # encoded idx
    behavior_id: int
    timestamp: np.datetime64
    sku: int      # original product id


def _group_events(df: pd.DataFrame) -> Dict[int, List[BehaviorEvent]]:
    out: Dict[int, List[BehaviorEvent]] = {}
    for user_id, group in df.groupby("client_id", sort=False):
        out[int(user_id)] = [
            BehaviorEvent(
                item_id=int(r.item_idx),
                behavior_id=int(r.behavior_id),
                timestamp=np.datetime64(r.timestamp.to_datetime64()),
                sku=int(r.sku),
            )
            for r in group.sort_values("timestamp").itertuples(index=False)
        ]
    return out


user_events_all = _group_events(events)
train_events_df = events[events["timestamp"] < VAL_CUTOFF]
user_events_train = _group_events(train_events_df)


def filter_purchase_only(seq: Sequence[BehaviorEvent]) -> List[BehaviorEvent]:
    return [e for e in seq if e.behavior_id == BEHAVIOR_BUY]


def history_before(
    seq: Sequence[BehaviorEvent],
    target_time: pd.Timestamp | np.datetime64,
    purchase_only: bool,
    max_len: int,
) -> List[BehaviorEvent]:
    t = np.datetime64(pd.Timestamp(target_time).to_datetime64())
    hist = [e for e in seq if e.timestamp < t]
    if purchase_only:
        hist = filter_purchase_only(hist)
    if len(hist) > max_len:
        hist = hist[-max_len:]
    return hist


def build_train_instances(purchase_only: bool, max_instances: Optional[int] = None) -> List[dict]:
    """Sliding next-buy targets from pre-val_cutoff timelines only."""
    instances: List[dict] = []

    if purchase_only:
        # Use exact Step1 train_sequences so purchase_only matches the baseline cohort/vocab.
        for user_id, skus in train_sequences.items():
            encoded = []
            for sku in skus:
                sku = int(sku)
                if sku not in item_to_idx:
                    continue
                encoded.append(
                    BehaviorEvent(
                        item_id=int(item_to_idx[sku]),
                        behavior_id=BEHAVIOR_BUY,
                        timestamp=np.datetime64("NaT"),
                        sku=sku,
                    )
                )
            for i in range(1, len(encoded)):
                hist = encoded[:i]
                if len(hist) > MAX_SEQ_LEN:
                    hist = hist[-MAX_SEQ_LEN:]
                target = encoded[i]
                instances.append(
                    {
                        "user_id": int(user_id),
                        "history": hist,
                        "target_item": int(target.item_id),
                        "target_sku": int(target.sku),
                    }
                )
    else:
        for user_id, seq in user_events_train.items():
            buy_positions = [i for i, e in enumerate(seq) if e.behavior_id == BEHAVIOR_BUY]
            for bpos in buy_positions:
                if bpos == 0:
                    continue
                target = seq[bpos]
                hist = seq[:bpos]
                if not hist:
                    continue
                if len(hist) > MAX_SEQ_LEN:
                    hist = hist[-MAX_SEQ_LEN:]
                instances.append(
                    {
                        "user_id": int(user_id),
                        "history": hist,
                        "target_item": int(target.item_id),
                        "target_sku": int(target.sku),
                    }
                )

    if max_instances is not None and len(instances) > max_instances:
        rng = np.random.default_rng(SEED)
        pick = rng.choice(len(instances), size=max_instances, replace=False)
        instances = [instances[i] for i in sorted(pick)]
    return instances


def resolve_target_time(
    seq: Sequence[BehaviorEvent],
    target_sku: int,
    window_start: pd.Timestamp,
    window_end: Optional[pd.Timestamp],
) -> Optional[np.datetime64]:
    """Recover the Step1 first-purchase-in-window timestamp for the target SKU."""
    start = np.datetime64(window_start.to_datetime64())
    end = None if window_end is None else np.datetime64(window_end.to_datetime64())
    for e in seq:
        if e.behavior_id != BEHAVIOR_BUY or e.sku != target_sku:
            continue
        if e.timestamp < start:
            continue
        if end is not None and e.timestamp >= end:
            continue
        return e.timestamp
    return None


def history_from_train_sequences(user: int) -> List[BehaviorEvent]:
    """Exact Step1 purchase history before val_cutoff (purchase_only path)."""
    skus = train_sequences.get(user) or train_sequences.get(int(user)) or []
    hist: List[BehaviorEvent] = []
    # Synthetic timestamps keep order; all are before VAL_CUTOFF by construction.
    base = np.datetime64(VAL_CUTOFF.to_datetime64()) - np.timedelta64(len(skus) + 1, "s")
    for i, sku in enumerate(skus):
        sku = int(sku)
        if sku not in item_to_idx:
            continue
        hist.append(
            BehaviorEvent(
                item_id=int(item_to_idx[sku]),
                behavior_id=BEHAVIOR_BUY,
                timestamp=base + np.timedelta64(i + 1, "s"),
                sku=sku,
            )
        )
    if len(hist) > MAX_SEQ_LEN:
        hist = hist[-MAX_SEQ_LEN:]
    return hist


def build_eval_examples(
    eval_users: Sequence[int],
    targets: Dict[int, int],
    purchase_only: bool,
    window_start: pd.Timestamp,
    window_end: Optional[pd.Timestamp],
) -> Dict[int, dict]:
    """One example per shared eval user; target SKU must match Step1 artifact."""
    examples: Dict[int, dict] = {}
    skipped = 0
    for user in eval_users:
        user = int(user)
        target_sku = int(targets[user])
        if target_sku not in item_to_idx:
            skipped += 1
            continue

        if purchase_only and window_start == VAL_CUTOFF and window_end == TEST_CUTOFF:
            # Validation: use exact Step1 train_sequences (pre-val_cutoff purchases)
            hist = history_from_train_sequences(user)
        else:
            seq = user_events_all.get(user, [])
            target_time = resolve_target_time(seq, target_sku, window_start, window_end)
            if target_time is None:
                skipped += 1
                continue
            hist = history_before(seq, target_time, purchase_only=purchase_only, max_len=MAX_SEQ_LEN)
            if purchase_only and not hist:
                # Fallback for test if event rebuild fails but Step1 train history exists
                hist = history_from_train_sequences(user)

        if not hist:
            skipped += 1
            continue
        examples[user] = {
            "user_id": user,
            "history": hist,
            "target_item": int(item_to_idx[target_sku]),
            "target_sku": target_sku,
        }
    if skipped:
        print(f"Skipped {skipped} eval users with empty/unresolvable history")
    return examples


print("Grouped users (all):", len(user_events_all))
print("Grouped users (train window):", len(user_events_train))


Grouped users (all): 504246
Grouped users (train window): 432906


## 5. Dataset, model, and shared-candidate evaluation


In [10]:
def _pad_or_truncate(
    item_ids: List[int],
    behavior_ids: List[int],
    max_len: int,
) -> Tuple[List[int], List[int]]:
    if len(item_ids) > max_len:
        item_ids = item_ids[-max_len:]
        behavior_ids = behavior_ids[-max_len:]
    pad = max_len - len(item_ids)
    if pad > 0:
        item_ids = [0] * pad + item_ids
        behavior_ids = [BEHAVIOR_PAD] * pad + behavior_ids
    return item_ids, behavior_ids


class SASRecDataset(Dataset):
    def __init__(
        self,
        examples: Sequence[dict],
        max_len: int,
        n_items: int,
        n_negatives: int = 1,
        seed: int = 42,
    ) -> None:
        self.examples = list(examples)
        self.max_len = max_len
        self.n_items = n_items
        self.n_negatives = n_negatives
        self.rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return len(self.examples)

    def _sample_negatives(self, positive: int, banned: set[int]) -> List[int]:
        negs: List[int] = []
        while len(negs) < self.n_negatives:
            neg = int(self.rng.integers(1, self.n_items + 1))
            if neg != positive and neg not in banned:
                negs.append(neg)
        return negs

    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        items = [e.item_id for e in ex["history"]]
        behaviors = [e.behavior_id for e in ex["history"]]
        items, behaviors = _pad_or_truncate(items, behaviors, self.max_len)
        out = {
            "user_id": torch.tensor(ex["user_id"], dtype=torch.long),
            "item_ids": torch.tensor(items, dtype=torch.long),
            "behavior_ids": torch.tensor(behaviors, dtype=torch.long),
            "target_item": torch.tensor(ex["target_item"], dtype=torch.long),
            "target_sku": torch.tensor(ex["target_sku"], dtype=torch.long),
        }
        if self.n_negatives > 0:
            banned = {e.item_id for e in ex["history"] if e.behavior_id == BEHAVIOR_BUY}
            banned.add(int(ex["target_item"]))
            negs = self._sample_negatives(int(ex["target_item"]), banned)
            out["neg_items"] = torch.tensor(negs, dtype=torch.long)
        return out


def collate_sasrec(batch: List[dict]) -> dict:
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in keys}


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 4, hidden_size),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False
        )
        x = x + self.dropout(attn_out)
        x = x + self.ff(self.norm2(x))
        return x


class SASRec(nn.Module):
    def __init__(
        self,
        n_items: int,
        hidden_size: int = 64,
        max_len: int = 50,
        n_blocks: int = 2,
        n_heads: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.n_items = n_items
        self.hidden_size = hidden_size
        self.max_len = max_len
        self.item_emb = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [SASRecBlock(hidden_size, n_heads, dropout) for _ in range(n_blocks)]
        )
        self.final_norm = nn.LayerNorm(hidden_size)
        nn.init.normal_(self.item_emb.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.item_emb.weight[0].fill_(0)

    def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        batch_size, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = self.item_emb(item_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)

    def predict(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
        candidate_items: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_emb = self.encode(item_ids, behavior_ids)
        lengths = (item_ids != 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        final = seq_emb[torch.arange(seq_emb.size(0), device=seq_emb.device), last_idx]
        if candidate_items is None:
            return torch.matmul(final, self.item_emb.weight.transpose(0, 1))
        cand_emb = self.item_emb(candidate_items)
        return (cand_emb * final.unsqueeze(1)).sum(dim=-1)


class BehaviorAwareSASRec(SASRec):
    def __init__(
        self,
        n_items: int,
        n_behaviors: int = 3,
        hidden_size: int = 64,
        max_len: int = 50,
        n_blocks: int = 2,
        n_heads: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__(
            n_items=n_items,
            hidden_size=hidden_size,
            max_len=max_len,
            n_blocks=n_blocks,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.behavior_emb = nn.Embedding(n_behaviors + 1, hidden_size, padding_idx=0)
        nn.init.normal_(self.behavior_emb.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.behavior_emb.weight[0].fill_(0)

    def encode(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if behavior_ids is None:
            raise ValueError("BehaviorAwareSASRec requires behavior_ids.")
        batch_size, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = self.item_emb(item_ids) + self.behavior_emb(behavior_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)


def bce_ranking_loss(
    model: SASRec,
    item_ids: torch.Tensor,
    behavior_ids: torch.Tensor,
    pos_items: torch.Tensor,
    neg_items: torch.Tensor,
) -> torch.Tensor:
    seq_emb = model.encode(item_ids, behavior_ids)
    lengths = (item_ids != 0).sum(dim=1).clamp(min=1)
    last_idx = lengths - 1
    final = seq_emb[torch.arange(seq_emb.size(0), device=seq_emb.device), last_idx]
    pos_emb = model.item_emb(pos_items)
    pos_logits = (final * pos_emb).sum(dim=-1)
    neg_emb = model.item_emb(neg_items)
    if neg_emb.dim() == 2:
        neg_logits = (final * neg_emb).sum(dim=-1)
        return -F.logsigmoid(pos_logits).mean() - F.logsigmoid(-neg_logits).mean()
    neg_logits = (neg_emb * final.unsqueeze(1)).sum(dim=-1)
    return -F.logsigmoid(pos_logits).mean() - F.logsigmoid(-neg_logits).mean()


@dataclass
class TrainConfig:
    hidden_size: int = 64
    max_len: int = 50
    n_blocks: int = 2
    n_heads: int = 1
    dropout: float = 0.2
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 0.0
    epochs: int = 20
    n_negatives: int = 1
    num_workers: int = 0
    seed: int = 42
    device: str = "cpu"
    patience: int = 5
    eval_k: int = 10


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model(mode: str, n_items: int, cfg: TrainConfig) -> SASRec:
    common = dict(
        n_items=n_items,
        hidden_size=cfg.hidden_size,
        max_len=cfg.max_len,
        n_blocks=cfg.n_blocks,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
    )
    if mode == "purchase_only":
        return SASRec(**common)
    if mode == "behavior_aware":
        return BehaviorAwareSASRec(n_behaviors=3, **common)
    raise ValueError(f"Unknown mode: {mode}")


@torch.no_grad()
def score_shared_candidates(
    model: SASRec,
    eval_examples: Dict[int, dict],
    candidate_sets: Dict[int, List[int]],
    cfg: TrainConfig,
) -> Dict[int, List[float]]:
    """Score Step2 candidate SKUs for each overlapping eval user."""
    model.eval()
    device = torch.device(cfg.device)
    users = [u for u in candidate_sets if u in eval_examples]
    scores_out: Dict[int, List[float]] = {}

    batch_users: List[int] = []
    batch_items: List[torch.Tensor] = []
    batch_behaviors: List[torch.Tensor] = []
    batch_cands: List[torch.Tensor] = []

    def flush() -> None:
        if not batch_users:
            return
        item_ids = torch.stack(batch_items).to(device)
        behavior_ids = torch.stack(batch_behaviors).to(device)
        cand_idx = torch.stack(batch_cands).to(device)
        logits = model.predict(item_ids, behavior_ids, candidate_items=cand_idx)
        logits_np = logits.detach().cpu().numpy()
        for i, user in enumerate(batch_users):
            scores_out[user] = logits_np[i].tolist()
        batch_users.clear()
        batch_items.clear()
        batch_behaviors.clear()
        batch_cands.clear()

    for user in users:
        ex = eval_examples[user]
        cands = candidate_sets[user]
        missing = [sku for sku in cands if int(sku) not in item_to_idx]
        if missing:
            raise ValueError(f"Candidate SKUs missing from vocab for user {user}: {missing[:5]}")
        cand_indices = [item_to_idx[int(sku)] for sku in cands]
        items = [e.item_id for e in ex["history"]]
        behaviors = [e.behavior_id for e in ex["history"]]
        items, behaviors = _pad_or_truncate(items, behaviors, cfg.max_len)
        batch_users.append(user)
        batch_items.append(torch.tensor(items, dtype=torch.long))
        batch_behaviors.append(torch.tensor(behaviors, dtype=torch.long))
        batch_cands.append(torch.tensor(cand_indices, dtype=torch.long))
        if len(batch_users) >= cfg.batch_size:
            flush()
    flush()
    return scores_out


def evaluate_on_shared_split(
    model: SASRec,
    eval_examples: Dict[int, dict],
    candidate_sets: Dict[int, List[int]],
    targets: Dict[int, int],
    cfg: TrainConfig,
) -> Tuple[dict, pd.DataFrame]:
    model_scores = score_shared_candidates(model, eval_examples, candidate_sets, cfg)
    cands = {u: candidate_sets[u] for u in model_scores}
    targs = {u: int(targets[u]) for u in model_scores}
    summary, user_results = evaluate_model_dataset(
        candidate_sets=cands,
        model_scores=model_scores,
        targets=targs,
        k=cfg.eval_k,
    )
    return summary, user_results


def format_metrics(summary: dict) -> str:
    parts = []
    for key in ("HitRate@10", "NDCG@10", "MRR", "n_users"):
        if key in summary:
            val = summary[key]
            parts.append(f"{key}={val:.4f}" if isinstance(val, float) else f"{key}={val}")
    return " | ".join(parts)


print("Model + shared-candidate eval helpers ready")


Model + shared-candidate eval helpers ready


## 6. Train config + build datasets


In [11]:
cfg = TrainConfig(
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_SEQ_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    n_negatives=N_NEGATIVES,
    seed=SEED,
    device=DEVICE,
    patience=PATIENCE,
    eval_k=EVAL_K,
)

train_instances_purchase = build_train_instances(
    purchase_only=True, max_instances=MAX_TRAIN_INSTANCES
)
train_instances_behavior = build_train_instances(
    purchase_only=False, max_instances=MAX_TRAIN_INSTANCES
)

val_examples_purchase = build_eval_examples(
    val_eval_users, val_targets, purchase_only=True,
    window_start=VAL_CUTOFF, window_end=TEST_CUTOFF,
)
test_examples_purchase = build_eval_examples(
    test_eval_users, test_targets, purchase_only=True,
    window_start=TEST_CUTOFF, window_end=None,
)
val_examples_behavior = build_eval_examples(
    val_eval_users, val_targets, purchase_only=False,
    window_start=VAL_CUTOFF, window_end=TEST_CUTOFF,
)
test_examples_behavior = build_eval_examples(
    test_eval_users, test_targets, purchase_only=False,
    window_start=TEST_CUTOFF, window_end=None,
)

print(cfg)
print(f"Train instances | purchase_only={len(train_instances_purchase):,} | behavior_aware={len(train_instances_behavior):,}")
print(
    f"Val coverage   | purchase_only={len(val_examples_purchase):,}/{len(val_eval_users):,} "
    f"| behavior_aware={len(val_examples_behavior):,}/{len(val_eval_users):,}"
)
print(
    f"Test coverage  | purchase_only={len(test_examples_purchase):,}/{len(test_eval_users):,} "
    f"| behavior_aware={len(test_examples_behavior):,}/{len(test_eval_users):,}"
)

for name, got, total in [
    ("val purchase", len(val_examples_purchase), len(val_eval_users)),
    ("test purchase", len(test_examples_purchase), len(test_eval_users)),
    ("val behavior", len(val_examples_behavior), len(val_eval_users)),
    ("test behavior", len(test_examples_behavior), len(test_eval_users)),
]:
    frac = got / total if total else 0
    if frac < 0.95:
        print(f"WARNING: {name} coverage {frac:.1%} — investigate history reconstruction")


TrainConfig(hidden_size=64, max_len=50, n_blocks=2, n_heads=1, dropout=0.2, batch_size=128, lr=0.001, weight_decay=0.0, epochs=20, n_negatives=1, num_workers=0, seed=42, device='cuda', patience=5, eval_k=10)
Train instances | purchase_only=622,354 | behavior_aware=837,681
Val coverage   | purchase_only=34,127/34,127 | behavior_aware=34,127/34,127
Test coverage  | purchase_only=35,637/35,637 | behavior_aware=35,637/35,637


## 7. Train + evaluate against shared popularity candidates


In [12]:
def train_model(
    mode: str,
    train_instances: List[dict],
    val_examples: Dict[int, dict],
    test_examples: Dict[int, dict],
    cfg: TrainConfig,
    output_dir: Path,
) -> dict:
    set_seed(cfg.seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(cfg.device)

    model = build_model(mode, n_items, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_ds = SASRecDataset(
        train_instances,
        max_len=cfg.max_len,
        n_items=n_items,
        n_negatives=cfg.n_negatives,
        seed=cfg.seed,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_sasrec,
        num_workers=cfg.num_workers,
    )

    history = []
    best_ndcg = -1.0
    best_path = output_dir / f"{mode}_best.pt"
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        losses = []
        for batch in tqdm(train_loader, desc=f"{mode} epoch {epoch}", leave=False):
            item_ids = batch["item_ids"].to(device)
            behavior_ids = batch["behavior_ids"].to(device)
            pos = batch["target_item"].to(device)
            neg = batch["neg_items"].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = bce_ranking_loss(model, item_ids, behavior_ids, pos, neg)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        val_summary, _ = evaluate_on_shared_split(
            model, val_examples, val_candidate_sets, val_targets, cfg
        )
        test_summary, _ = evaluate_on_shared_split(
            model, test_examples, test_candidate_sets, test_targets, cfg
        )
        row = {
            "epoch": epoch,
            "loss": float(np.mean(losses)) if losses else None,
            "val": val_summary,
            "test": test_summary,
        }
        history.append(row)
        print(
            f"[{mode}] epoch={epoch} loss={row['loss']:.4f} "
            f"val({format_metrics(val_summary)}) test({format_metrics(test_summary)})"
        )

        ndcg = float(val_summary.get("NDCG@10", 0.0))
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "mode": mode,
                    "n_items": n_items,
                    "config": asdict(cfg),
                    "val_metrics": val_summary,
                    "test_metrics": test_summary,
                    "protocol": {
                        "sequences_path": str(SEQUENCES_PATH),
                        "candidates_path": str(CANDIDATES_PATH),
                        "max_seq_len": MAX_SEQ_LEN,
                        "n_eval_negatives": N_EVAL_NEGATIVES,
                        "val_cutoff": str(VAL_CUTOFF),
                        "test_cutoff": str(TEST_CUTOFF),
                        "seed": SEED,
                    },
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{mode}] early stopping at epoch {epoch}")
                break

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    final_val, val_user_results = evaluate_on_shared_split(
        model, val_examples, val_candidate_sets, val_targets, cfg
    )
    final_test, test_user_results = evaluate_on_shared_split(
        model, test_examples, test_candidate_sets, test_targets, cfg
    )
    val_user_results.to_csv(output_dir / f"{mode}_val_user_results.csv", index=False)
    test_user_results.to_csv(output_dir / f"{mode}_test_user_results.csv", index=False)
    return {
        "mode": mode,
        "best_checkpoint": str(best_path),
        "history": history,
        "val_metrics": final_val,
        "test_metrics": final_test,
        "config": asdict(cfg),
    }


modes = ["purchase_only", "behavior_aware"] if MODE == "both" else [MODE]
results = {}
for mode in modes:
    print("=" * 72)
    print("Training", mode)
    if mode == "purchase_only":
        results[mode] = train_model(
            mode,
            train_instances_purchase,
            val_examples_purchase,
            test_examples_purchase,
            cfg,
            OUTPUT_DIR,
        )
    else:
        results[mode] = train_model(
            mode,
            train_instances_behavior,
            val_examples_behavior,
            test_examples_behavior,
            cfg,
            OUTPUT_DIR,
        )
    print(f"[{mode}] final val :", format_metrics(results[mode]["val_metrics"]))
    print(f"[{mode}] final test:", format_metrics(results[mode]["test_metrics"]))


Training purchase_only


purchase_only epoch 1:   0%|          | 0/4863 [00:00<?, ?it/s]

[purchase_only] epoch=1 loss=1.2662 val(HitRate@10=0.3422 | NDCG@10=0.2311 | MRR=0.2118 | n_users=34127) test(HitRate@10=0.2655 | NDCG@10=0.1776 | MRR=0.1660 | n_users=35637)


purchase_only epoch 2:   0%|          | 0/4863 [00:00<?, ?it/s]

[purchase_only] epoch=2 loss=1.1390 val(HitRate@10=0.3496 | NDCG@10=0.2343 | MRR=0.2132 | n_users=34127) test(HitRate@10=0.2675 | NDCG@10=0.1777 | MRR=0.1653 | n_users=35637)


purchase_only epoch 3:   0%|          | 0/4863 [00:00<?, ?it/s]

... (17 outputs left out)

behavior_aware epoch 5:   0%|          | 0/6545 [00:00<?, ?it/s]

[behavior_aware] epoch=5 loss=0.7592 val(HitRate@10=0.3388 | NDCG@10=0.2144 | MRR=0.1924 | n_users=34127) test(HitRate@10=0.2635 | NDCG@10=0.1660 | MRR=0.1526 | n_users=35637)


behavior_aware epoch 6:   0%|          | 0/6545 [00:00<?, ?it/s]

[behavior_aware] epoch=6 loss=0.6822 val(HitRate@10=0.3318 | NDCG@10=0.2048 | MRR=0.1824 | n_users=34127) test(HitRate@10=0.2574 | NDCG@10=0.1592 | MRR=0.1459 | n_users=35637)
[behavior_aware] early stopping at epoch 6
[behavior_aware] final val : HitRate@10=0.3812 | NDCG@10=0.2553 | MRR=0.2312 | n_users=34127
[behavior_aware] final test: HitRate@10=0.3061 | NDCG@10=0.2003 | MRR=0.1837 | n_users=35637


## 8. Compare with popularity baseline (same val users / candidates)


In [13]:
rows = []
for mode, result in results.items():
    for split_name in ("val", "test"):
        m = result[f"{split_name}_metrics"]
        rows.append(
            {
                "model": mode,
                "split": split_name,
                "n_users": m.get("n_users"),
                "HitRate@10": m.get("HitRate@10"),
                "NDCG@10": m.get("NDCG@10"),
                "MRR": m.get("MRR"),
            }
        )

comparison = pd.DataFrame(rows)

pop_summary_path = ARTIFACT_DIR / "popularity_val_summary.csv"
if pop_summary_path.exists():
    pop = pd.read_csv(pop_summary_path)
    pop_row = {
        "model": "popularity",
        "split": "val",
        "n_users": int(pop.loc[0, "n_users"]),
        "HitRate@10": float(pop.loc[0, "HitRate@10"]),
        "NDCG@10": float(pop.loc[0, "NDCG@10"]),
        "MRR": float(pop.loc[0, "MRR"]),
    }
    comparison = pd.concat([pd.DataFrame([pop_row]), comparison], ignore_index=True)

comparison_path = OUTPUT_DIR / "comparison_with_popularity.csv"
comparison.to_csv(comparison_path, index=False)

summary_path = OUTPUT_DIR / "summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "protocol": {
                "sequences_path": str(SEQUENCES_PATH),
                "candidates_path": str(CANDIDATES_PATH),
                "max_seq_len": MAX_SEQ_LEN,
                "n_eval_negatives": N_EVAL_NEGATIVES,
                "eval_k": EVAL_K,
                "val_cutoff": str(VAL_CUTOFF),
                "test_cutoff": str(TEST_CUTOFF),
                "n_items": n_items,
                "seed": SEED,
            },
            "results": {
                mode: {
                    "val": result["val_metrics"],
                    "test": result["test_metrics"],
                    "best_checkpoint": result["best_checkpoint"],
                }
                for mode, result in results.items()
            },
        },
        f,
        indent=2,
        default=float,
    )

print("Wrote", comparison_path)
print("Wrote", summary_path)
comparison


Wrote /content/artifacts/sasrec_aligned/comparison_with_popularity.csv
Wrote /content/artifacts/sasrec_aligned/summary.json


,model,split,n_users,HitRate@10,NDCG@10,MRR
0,popularity,val,34127,0.388226,0.256578,0.228811
1,purchase_only,val,34127,0.349635,0.234304,0.213162
2,purchase_only,test,35637,0.267475,0.177723,0.165281
3,behavior_aware,val,34127,0.381194,0.255345,0.231242
4,behavior_aware,test,35637,0.306086,0.200328,0.183661
